# 🔍 CNN 기반 불량 분류 (Defect Classification)

> **학습 목표**: 딥러닝 CNN으로 제조 불량 유형을 자동 분류합니다.

---

## 📋 학습 내용

1. ✅ 불량 이미지 데이터셋 준비
2. ✅ MobileNetV2 전이학습 (Transfer Learning)
3. ✅ 모델 학습 및 검증
4. ✅ 혼동 행렬 (Confusion Matrix) 분석
5. ✅ 분류 리포트 (Precision, Recall, F1-Score)
6. ✅ 예측 결과 시각화

**소요 시간**: 약 45분  
**난이도**: ⭐⭐⭐ (고급)  
**사전 지식**: CNN 기초, 전이학습 개념

---

## 🔧 Step 1: 라이브러리 Import

> 이 실습은 **TensorFlow/Keras (MobileNetV2)** 와 **PyTorch (ResNet18)** 두 경로를 모두 지원합니다.
> 설치된 프레임워크에 따라 자동으로 전환됩니다.

In [ ]:
# ============================================================
# TensorFlow/Keras 시도
# ============================================================
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    print(f"✅ TensorFlow 버전: {tf.__version__}")
    FRAMEWORK = 'tensorflow'
except ImportError:
    print("⚠️ TensorFlow 미설치")
    FRAMEWORK = None

# ============================================================
# PyTorch 시도 (TF 없을 경우 대안)
# ============================================================
if FRAMEWORK is None:
    try:
        import torch
        import torch.nn as nn
        import torchvision
        import torchvision.models as tv_models
        from torchvision import transforms
        from torch.utils.data import DataLoader, TensorDataset
        print(f"✅ PyTorch 버전: {torch.__version__}")
        FRAMEWORK = 'pytorch'
    except ImportError:
        print("⚠️ PyTorch 미설치")

if FRAMEWORK is None:
    print("\n❌ TensorFlow 또는 PyTorch 중 하나를 설치하세요.")
    print("   TF:      pip install tensorflow")
    print("   PyTorch: pip install torch torchvision")
else:
    print(f"\n✅ 사용 프레임워크: {FRAMEWORK.upper()}")

# ============================================================
# 공통 라이브러리
# ============================================================
import pandas as pd
import numpy as np
from PIL import Image
try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False
    print("⚠️ opencv-python 미설치 (Grad-CAM 오버레이 제한)")

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)

# GPU 확인
if FRAMEWORK == 'tensorflow':
    gpus = tf.config.list_physical_devices('GPU')
    print(f"{'✅ GPU' if gpus else 'ℹ️ CPU'} 모드로 실행 (GPU: {len(gpus) if gpus else 0}개)")
elif FRAMEWORK == 'pytorch':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"{'✅ GPU(CUDA)' if device.type == 'cuda' else 'ℹ️ CPU'} 모드로 실행")

print("\n✅ 라이브러리 로드 완료!")

## 📂 Step 2: 불량 데이터셋 준비

In [ ]:
def create_synthetic_defect_data(n_samples=1000, img_size=64):
    """
    합성 불량/정상 이미지 데이터 생성

    KAMP 실제 데이터가 없을 경우 사용하는 대체 데이터입니다.
    불량(1)은 이미지에 흰색 점(스크래치 시뮬레이션)을 추가합니다.

    Parameters
    ----------
    n_samples : 생성할 총 샘플 수
    img_size  : 이미지 크기 (img_size × img_size)

    Returns
    -------
    images : ndarray (n_samples, img_size, img_size, 3) float32 [0,1]
    labels : ndarray (n_samples,) int — 0=정상, 1=불량
    """
    np.random.seed(42)
    images = []
    labels = []

    for i in range(n_samples):
        # 기본 배경 (금속 표면 질감 시뮬레이션)
        img = np.random.rand(img_size, img_size, 3).astype(np.float32) * 0.3
        # 약간의 그레인(grain) 효과
        noise = np.random.randn(img_size, img_size, 3).astype(np.float32) * 0.05
        img = np.clip(img + noise, 0, 1)

        if i % 2 == 0:
            # 정상 — 균일한 배경
            labels.append(0)
        else:
            # 불량 — 스크래치/오염 시뮬레이션
            defect_type = np.random.choice(['scratch', 'spot', 'line'])
            if defect_type == 'scratch':
                # 선형 스크래치
                x1 = np.random.randint(5, img_size - 15)
                y1 = np.random.randint(5, img_size - 15)
                length = np.random.randint(8, 20)
                for k in range(length):
                    r = min(x1 + k, img_size - 1)
                    c = min(y1 + k // 2, img_size - 1)
                    img[r, c] = [0.95, 0.95, 0.95]
            elif defect_type == 'spot':
                # 점형 오염
                cx = np.random.randint(10, img_size - 10)
                cy = np.random.randint(10, img_size - 10)
                r = np.random.randint(3, 7)
                y_idx, x_idx = np.ogrid[-r:r+1, -r:r+1]
                mask = x_idx**2 + y_idx**2 <= r**2
                cx1, cx2 = max(0, cx-r), min(img_size, cx+r+1)
                cy1, cy2 = max(0, cy-r), min(img_size, cy+r+1)
                img[cy1:cy2, cx1:cx2][mask[:cy2-cy1, :cx2-cx1]] = [0.9, 0.9, 0.9]
            else:
                # 수평선형 결함
                row = np.random.randint(10, img_size - 10)
                img[row, img_size//4:3*img_size//4] = [0.85, 0.85, 0.85]
            labels.append(1)

        images.append(img)

    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int64)


# ============================================================
# 데이터 디렉토리 확인 및 데이터 준비
# ============================================================
kamp_dir    = Path('../../dataset/part1-2/시지트로닉스_이미지')
fallback_dir = Path('../data/defect_images')

if kamp_dir.exists() and any(d.is_dir() for d in kamp_dir.iterdir()):
    data_dir = kamp_dir
    USE_SYNTHETIC = False
    print(f"✅ KAMP 실제 데이터 사용: {data_dir.resolve()}")
else:
    data_dir = fallback_dir
    USE_SYNTHETIC = not (data_dir.exists() and
                         any(d.is_dir() for d in data_dir.iterdir()
                             if (d / next(iter(d.glob('*.jpg').__class__.__iter__(d.glob('*.jpg'))),
                                          d / '_.png')).exists())
                         if data_dir.exists() else True)
    if USE_SYNTHETIC:
        print("ℹ️ KAMP / 로컬 이미지 데이터 없음 → 합성 데이터로 학습합니다.")
    else:
        print(f"ℹ️ 로컬 샘플 데이터 사용: {data_dir}")

# 데이터셋 구조 확인 및 폴더 생성
if not data_dir.exists():
    data_dir.mkdir(exist_ok=True, parents=True)

classes = sorted([d.name for d in data_dir.iterdir() if d.is_dir()]) if data_dir.exists() else []

if len(classes) == 0:
    # 클래스 폴더 없을 경우 샘플 이미지 생성
    classes = ['normal', 'scratch', 'contamination', 'damage']
    print(f"📁 샘플 이미지 폴더 생성 중: {classes}")
    for cls in classes:
        cls_dir = data_dir / cls
        cls_dir.mkdir(exist_ok=True)
        for i in range(20):  # 클래스당 20개
            img_arr = (np.random.rand(224, 224, 3) * 255).astype(np.uint8)
            if cls != 'normal':
                # 불량 시뮬레이션: 흰 점 추가
                dx, dy = np.random.randint(50, 174, 2)
                img_arr[dx:dx+10, dy:dy+10] = 255
            Image.fromarray(img_arr).save(cls_dir / f'{cls}_{i:03d}.png')
    print(f"✅ 샘플 이미지 생성 완료 (클래스당 20개)")

print(f"\n📊 데이터셋 구조:")
class_counts = {}
for cls in classes:
    cls_dir = data_dir / cls
    imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    class_counts[cls] = len(imgs)
    print(f"   - {cls}: {len(imgs)}개")

total_images = sum(class_counts.values())
print(f"\n✅ 총 이미지: {total_images}개 | 클래스: {len(classes)}개")

In [ ]:
# 샘플 이미지 시각화
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, cls in enumerate(classes):
    cls_dir = data_dir / cls
    images = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    
    if len(images) > 0:
        sample_img = Image.open(images[0])
        axes[idx].imshow(sample_img)
        axes[idx].set_title(f'{cls} ({class_counts[cls]}개)', fontsize=12, fontweight='bold')
        axes[idx].axis('off')
    else:
        axes[idx].text(0.5, 0.5, 'No Image', ha='center', va='center')
        axes[idx].set_title(cls, fontsize=12, fontweight='bold')
        axes[idx].axis('off')

# 빈 서브플롯 숨기기
for idx in range(len(classes), 8):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 🤖 Step 3: MobileNetV2 전이학습 모델 구축

In [ ]:
if FRAMEWORK == 'tensorflow':
    # 하이퍼파라미터
    IMG_SIZE = (224, 224)
    BATCH_SIZE = 16
    EPOCHS = 10
    NUM_CLASSES = len(classes)
    
    print(f"⚙️ 하이퍼파라미터:")
    print(f"   이미지 크기: {IMG_SIZE}")
    print(f"   배치 크기: {BATCH_SIZE}")
    print(f"   에폭: {EPOCHS}")
    print(f"   클래스 수: {NUM_CLASSES}")
    
    # 데이터 증강 (Data Augmentation)
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        validation_split=0.2  # 20% 검증 데이터
    )
    
    # 학습 데이터 생성
    train_generator = train_datagen.flow_from_directory(
        data_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training'
    )
    
    # 검증 데이터 생성
    val_generator = train_datagen.flow_from_directory(
        data_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation'
    )
    
    print(f"\n✅ 데이터 생성기 준비 완료")
    print(f"   학습 샘플: {train_generator.samples}개")
    print(f"   검증 샘플: {val_generator.samples}개")
    print(f"   클래스 매핑: {train_generator.class_indices}")
else:
    print("⚠️ TensorFlow가 설치되지 않아 모델 구축을 건너뜁니다.")

In [ ]:
if FRAMEWORK == 'tensorflow':
    # MobileNetV2 기본 모델 (ImageNet 가중치)
    print("🔄 MobileNetV2 모델 로드 중...")
    
    base_model = MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,  # 분류 헤드 제거
        weights='imagenet'
    )
    
    # 기본 모델 가중치 동결 (Fine-tuning 전략)
    base_model.trainable = False
    
    # 분류 헤드 추가
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    
    # 모델 컴파일
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    print(f"\n✅ 모델 구축 완료!")
    print(f"   총 파라미터: {model.count_params():,}개")
    print(f"   학습 가능 파라미터: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}개")
    
    # 모델 구조 요약
    model.summary()
else:
    print("⚠️ TensorFlow가 설치되지 않아 모델 생성을 건너뜁니다.")

---

## 🔥 Step 3-B: PyTorch 경로 — ResNet18 전이학습 (TF 미설치 시 실행)

TensorFlow가 없는 환경을 위한 PyTorch ResNet18 기반 CNN 분류 모델입니다.
합성 데이터를 사용해 TF 없이도 전이학습 전 과정을 실습할 수 있습니다.

In [ ]:
if FRAMEWORK == 'pytorch':
    print("🔥 PyTorch ResNet18 전이학습 경로 실행\n")

    # ============================================================
    # 합성 이미지 데이터 생성 (이진 분류: 정상/불량)
    # ============================================================
    IMG_SIZE_PT   = 64     # ResNet 입력 크기 (경량화)
    N_SAMPLES_PT  = 1000
    BATCH_SIZE_PT = 32
    EPOCHS_PT     = 10
    NUM_CLASSES_PT = 2

    print(f"⚙️ 하이퍼파라미터 (PyTorch):")
    print(f"   이미지 크기: {IMG_SIZE_PT}×{IMG_SIZE_PT}")
    print(f"   배치 크기:   {BATCH_SIZE_PT}")
    print(f"   에폭:        {EPOCHS_PT}")
    print(f"   클래스 수:   {NUM_CLASSES_PT} (정상/불량)")

    syn_images, syn_labels = create_synthetic_defect_data(N_SAMPLES_PT, IMG_SIZE_PT)
    print(f"\n✅ 합성 데이터 생성: {syn_images.shape} (float32, [0,1])")
    print(f"   불량 비율: {syn_labels.mean()*100:.1f}%")

    # ============================================================
    # 샘플 시각화
    # ============================================================
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    label_names_pt = ['정상(0)', '불량(1)']

    for i, ax in enumerate(axes.flatten()):
        idx = i * 2
        if idx < len(syn_images):
            ax.imshow(syn_images[idx])
            ax.set_title(f'{label_names_pt[syn_labels[idx]]}', fontsize=10, fontweight='bold',
                         color='green' if syn_labels[idx] == 0 else 'red')
        ax.axis('off')

    plt.suptitle('합성 데이터 샘플 — 정상 vs 불량 시뮬레이션', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ============================================================
    # Train/Val/Test 분리
    # ============================================================
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        syn_images, syn_labels, test_size=0.3, random_state=42, stratify=syn_labels
    )
    X_val, X_te, y_val, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp
    )
    print(f"\n📦 분할: Train={len(X_tr)}, Val={len(X_val)}, Test={len(X_te)}")

    # PyTorch Tensor 변환 (NHWC → NCHW)
    to_tensor = lambda x, y: (
        torch.from_numpy(x.transpose(0, 3, 1, 2)).float(),
        torch.from_numpy(y).long()
    )
    X_tr_t, y_tr_t   = to_tensor(X_tr,  y_tr)
    X_val_t, y_val_t = to_tensor(X_val, y_val)
    X_te_t,  y_te_t  = to_tensor(X_te,  y_te)

    train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t),   batch_size=BATCH_SIZE_PT, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE_PT)
    test_loader  = DataLoader(TensorDataset(X_te_t,  y_te_t),  batch_size=BATCH_SIZE_PT)

    # ============================================================
    # ResNet18 전이학습 모델 정의
    # ============================================================
    pt_model = tv_models.resnet18(pretrained=False)

    # 마지막 FC 레이어를 2클래스(정상/불량)로 교체
    num_features = pt_model.fc.in_features
    pt_model.fc = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, NUM_CLASSES_PT)   # 정상/불량
    )
    pt_model = pt_model.to(device)

    total_params     = sum(p.numel() for p in pt_model.parameters())
    trainable_params = sum(p.numel() for p in pt_model.parameters() if p.requires_grad)
    print(f"\n✅ ResNet18 모델 구성 완료")
    print(f"   총 파라미터:     {total_params:,}개")
    print(f"   학습 가능 파라미터: {trainable_params:,}개")
    print(f"   마지막 레이어: FC({num_features}) → 256 → {NUM_CLASSES_PT}")

else:
    print(f"ℹ️ {FRAMEWORK.upper()} 프레임워크 사용 중 — PyTorch 경로 건너뜁니다.")
    print("   (TF 경로는 Step 3 / Step 4에서 실행됩니다.)")

In [ ]:
if FRAMEWORK == 'pytorch':
    # ============================================================
    # PyTorch 학습 루프
    # ============================================================
    def train_model_pytorch(model, train_loader, val_loader, epochs=10):
        """
        PyTorch 학습 루프

        Returns
        -------
        history : dict with 'train_loss', 'val_loss', 'val_acc'
        """
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

        history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
        best_val_acc = 0.0
        best_state = None

        print(f"🔄 학습 시작 (epochs={epochs}, device={device})\n")
        for epoch in range(epochs):
            # --- 학습 ---
            model.train()
            train_loss = 0.0
            for inputs, labels_b in train_loader:
                inputs, labels_b = inputs.to(device), labels_b.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels_b)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            # --- 검증 ---
            model.eval()
            val_loss = 0.0
            correct = total = 0
            with torch.no_grad():
                for inputs, labels_b in val_loader:
                    inputs, labels_b = inputs.to(device), labels_b.to(device)
                    outputs = model(inputs)
                    val_loss += criterion(outputs, labels_b).item()
                    preds = outputs.argmax(dim=1)
                    correct += (preds == labels_b).sum().item()
                    total   += labels_b.size(0)

            t_loss = train_loss / len(train_loader)
            v_loss = val_loss   / len(val_loader)
            v_acc  = correct / total

            history['train_loss'].append(t_loss)
            history['val_loss'].append(v_loss)
            history['val_acc'].append(v_acc)

            # Best 모델 저장
            if v_acc > best_val_acc:
                best_val_acc = v_acc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

            scheduler.step()

            if (epoch + 1) % 2 == 0 or epoch == 0:
                print(f"   Epoch {epoch+1:2d}/{epochs} | "
                      f"Train Loss: {t_loss:.4f} | "
                      f"Val Loss: {v_loss:.4f} | "
                      f"Val Acc: {v_acc:.4f}")

        # Best 모델 가중치 복원
        if best_state:
            model.load_state_dict(best_state)
        print(f"\n✅ 학습 완료! Best Val Accuracy: {best_val_acc:.4f}")
        return history, model

    pt_history, pt_model = train_model_pytorch(
        pt_model, train_loader, val_loader, epochs=EPOCHS_PT
    )

    # ============================================================
    # 학습 곡선 시각화
    # ============================================================
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epochs_range = range(1, len(pt_history['train_loss']) + 1)

    axes[0].plot(epochs_range, pt_history['train_loss'], label='Train Loss', linewidth=2, color='#0072B2')
    axes[0].plot(epochs_range, pt_history['val_loss'],   label='Val Loss',   linewidth=2, color='#D55E00')
    axes[0].set_title('학습 곡선 — Loss', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_range, pt_history['val_acc'], label='Val Accuracy', linewidth=2,
                 color='#009E73', marker='o', markersize=4)
    axes[1].set_title('학습 곡선 — Validation Accuracy', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1.05)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # ============================================================
    # Test 세트 평가
    # ============================================================
    pt_model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for inputs, labels_b in test_loader:
            inputs = inputs.to(device)
            outputs = pt_model(inputs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_b.numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    pt_acc = accuracy_score(all_labels, all_preds)
    print(f"\n📊 Test Accuracy: {pt_acc:.4f}")

    # Confusion Matrix
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    cm_pt = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm_pt, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['정상(0)', '불량(1)'],
                yticklabels=['정상(0)', '불량(1)'])
    axes[0].set_title('Confusion Matrix (PyTorch ResNet18)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('예측 클래스')
    axes[0].set_ylabel('실제 클래스')

    # 클래스별 성능 (분류 리포트 시각화)
    report_pt = classification_report(all_labels, all_preds,
                                       target_names=['정상', '불량'],
                                       output_dict=True, zero_division=0)
    metrics_pt = pd.DataFrame(report_pt).transpose().loc[['정상', '불량'],
                                                          ['precision', 'recall', 'f1-score']]
    metrics_pt.plot(kind='bar', ax=axes[1], width=0.6)
    axes[1].set_title('클래스별 성능 지표', fontsize=12, fontweight='bold')
    axes[1].set_xticklabels(['정상', '불량'], rotation=0)
    axes[1].set_ylim(0, 1.1)
    axes[1].legend(title='지표')
    axes[1].grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

    print("\n📋 분류 리포트:")
    print(classification_report(all_labels, all_preds,
                                 target_names=['정상', '불량'], zero_division=0))

    # 모델 저장
    import os
    os.makedirs('../outputs', exist_ok=True)
    torch.save(pt_model.state_dict(), '../outputs/defect_classifier_resnet18.pth')
    print("✅ PyTorch 모델 저장: ../outputs/defect_classifier_resnet18.pth")

    # history 및 결과 저장
    pt_model_saved = pt_model   # 이후 시각화 셀에서 참조
    pt_all_preds   = all_preds
    pt_all_labels  = all_labels
    pt_syn_images  = syn_images
    pt_syn_labels  = syn_labels

else:
    print(f"ℹ️ {FRAMEWORK.upper() if FRAMEWORK else '없음'} 프레임워크 — PyTorch 학습 루프 건너뜁니다.")

## 🎓 Step 4: 모델 학습

In [ ]:
if FRAMEWORK == 'tensorflow' and train_generator.samples > 0:
    # 콜백 설정
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        ),
        ModelCheckpoint(
            '../outputs/best_defect_model.h5',
            monitor='val_accuracy',
            save_best_only=True
        )
    ]
    
    print("🔄 모델 학습 시작...")
    print(f"   에폭: {EPOCHS}")
    print(f"   Early Stopping: patience=3")
    
    # 학습
    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=val_generator,
        callbacks=callbacks,
        verbose=1
    )
    
    print("\n✅ 모델 학습 완료!")
else:
    print("⚠️ 학습 데이터가 부족하거나 TensorFlow가 없어 학습을 건너뜁니다.")
    history = None

In [ ]:
if history is not None:
    # 학습 곡선 시각화
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
    axes[0].set_title('모델 정확도 (Accuracy)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Loss
    axes[1].plot(history.history['loss'], label='Train', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
    axes[1].set_title('모델 손실 (Loss)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 최종 성능
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    
    print(f"\n📊 최종 성능:")
    print(f"   Train Accuracy: {final_train_acc:.4f}")
    print(f"   Validation Accuracy: {final_val_acc:.4f}")
    
    if final_train_acc - final_val_acc > 0.1:
        print(f"\n⚠️ 과적합 가능성 (Train-Val 차이: {final_train_acc - final_val_acc:.4f})")
    else:
        print(f"\n✅ 적절한 일반화 성능")
else:
    print("⚠️ 학습 기록이 없어 시각화를 건너뜁니다.")

## 📊 Step 5: 모델 평가 (혼동 행렬)

In [ ]:
if FRAMEWORK == 'tensorflow' and history is not None:
    # 검증 데이터에 대한 예측
    print("🔄 모델 예측 중...")
    
    val_generator.reset()
    y_pred_probs = model.predict(val_generator)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = val_generator.classes
    
    # 정확도
    accuracy = accuracy_score(y_true, y_pred)
    print(f"\n📊 검증 세트 정확도: {accuracy:.4f}")
    
    # 혼동 행렬
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': '샘플 수'})
    plt.title('혼동 행렬 (Confusion Matrix)', fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('예측 클래스')
    plt.ylabel('실제 클래스')
    plt.tight_layout()
    plt.show()
    
    print("\n💡 혼동 행렬 해석:")
    print("   - 대각선: 올바르게 분류된 샘플")
    print("   - 비대각선: 잘못 분류된 샘플")
    print("   - 행: 실제 클래스")
    print("   - 열: 예측 클래스")
else:
    print("⚠️ 모델이 없어 평가를 건너뜁니다.")

## 📋 Step 6: 분류 리포트 (Precision, Recall, F1-Score)

In [ ]:
if FRAMEWORK == 'tensorflow' and history is not None:
    # 분류 리포트
    report = classification_report(y_true, y_pred, target_names=classes, output_dict=True)
    
    # DataFrame 변환
    report_df = pd.DataFrame(report).transpose()
    
    print("📋 분류 리포트:")
    print(report_df.to_string())
    
    # 클래스별 성능 시각화
    metrics = ['precision', 'recall', 'f1-score']
    class_metrics = report_df.loc[classes, metrics]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    class_metrics.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title('클래스별 성능 지표', fontsize=14, fontweight='bold')
    ax.set_xlabel('클래스')
    ax.set_ylabel('점수')
    ax.set_xticklabels(classes, rotation=45, ha='right')
    ax.legend(title='지표', loc='lower right')
    ax.grid(alpha=0.3, axis='y')
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    plt.show()
    
    print("\n💡 성능 지표 의미:")
    print("   - Precision (정밀도): 예측이 얼마나 정확한가?")
    print("   - Recall (재현율): 실제를 얼마나 잘 찾았는가?")
    print("   - F1-Score: Precision과 Recall의 조화평균")
    print("   - Support: 각 클래스의 실제 샘플 수")
else:
    print("⚠️ 모델이 없어 리포트를 건너뜁니다.")

## 🖼️ Step 7: 예측 결과 시각화

In [ ]:
if FRAMEWORK == 'tensorflow' and history is not None:
    # 샘플 예측 시각화
    num_samples = min(8, len(y_pred))
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    val_generator.reset()
    sample_batch = next(val_generator)
    
    for i in range(num_samples):
        img = sample_batch[0][i]
        true_label = classes[y_true[i]]
        pred_label = classes[y_pred[i]]
        confidence = y_pred_probs[i][y_pred[i]] * 100
        
        axes[i].imshow(img)
        
        # 제목 색상 (정답/오답)
        color = 'green' if y_true[i] == y_pred[i] else 'red'
        title = f'실제: {true_label}\n예측: {pred_label} ({confidence:.1f}%)'
        axes[i].set_title(title, fontsize=10, fontweight='bold', color=color)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # 정답/오답 통계
    correct = (y_true == y_pred).sum()
    incorrect = len(y_true) - correct
    
    print(f"\n📊 예측 통계:")
    print(f"   정답: {correct}개 ({correct/len(y_true)*100:.1f}%)")
    print(f"   오답: {incorrect}개 ({incorrect/len(y_true)*100:.1f}%)")
else:
    print("⚠️ 모델이 없어 예측 시각화를 건너뜁니다.")

## 🔍 Step 7-B: Grad-CAM — CNN이 불량을 판단한 근거 시각화

> **학습 목표**: AI가 이미지 어느 부분을 보고 판단했는지 시각화합니다.

**왜 Grad-CAM이 필요한가?**

공장 현장에서 "AI가 이 제품을 불량이라고 판단했는데, 왜 그런가?"를 설명할 수 있어야 합니다.  
Grad-CAM은 CNN의 마지막 합성곱 레이어의 gradient를 계산해  
판단에 가장 크게 기여한 영역을 히트맵으로 표시합니다.

```
입력 이미지
    ↓
[Conv 레이어들] → 특징 맵 추출
    ↓
[마지막 Conv 레이어] ← Gradient 계산 (Grad-CAM 핵심)
    ↓
채널별 중요도 가중 평균 → 히트맵 생성
    ↓
원본 이미지 오버레이 → 판단 근거 시각화
```


In [ ]:

# ============================================================
# 🔍 Grad-CAM 시각화 (외부 라이브러리 없이 TF 2.x 순수 구현)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

# 한글 폰트 설정
def _setup_font():
    try:
        import google.colab
        import subprocess
        subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
        fm._load_fontmanager(try_read_cache=False)
        plt.rcParams['font.family'] = 'NanumGothic'
    except ImportError:
        sys = platform.system()
        if sys == 'Darwin':   plt.rcParams['font.family'] = 'AppleGothic'
        elif sys == 'Linux':
            nf = [f for f in fm.fontManager.ttflist if 'Nanum' in f.name]
            if nf: plt.rcParams['font.family'] = 'NanumGothic'
        elif sys == 'Windows': plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False

_setup_font()

def compute_gradcam_tf(model, img_array, class_idx=None):
    """
    Grad-CAM 히트맵 계산 (TensorFlow/Keras, 외부 패키지 불필요)

    Args:
        model:     학습된 Keras 모델
        img_array: 정규화된 입력 이미지 (1, H, W, 3)
        class_idx: 시각화할 클래스 인덱스 (None=예측 클래스)

    Returns:
        heatmap (H, W), pred_class_idx (int)
    """
    import tensorflow as tf

    # 마지막 Conv2D 레이어 자동 탐지
    last_conv_name = None
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            last_conv_name = layer.name
            break
        if hasattr(layer, 'layers'):          # 중첩 모델 (MobileNetV2 등)
            for sub in reversed(layer.layers):
                if isinstance(sub, tf.keras.layers.Conv2D):
                    last_conv_name = sub.name
                    break
            if last_conv_name:
                break

    if last_conv_name is None:
        print("⚠️ Conv2D 레이어를 찾지 못했습니다.")
        return None, None

    print(f"   Grad-CAM 기준 레이어: {last_conv_name}")

    # Gradient 계산용 서브모델
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if class_idx is None:
            class_idx = int(tf.argmax(preds[0]))
        score = preds[:, class_idx]

    grads       = tape.gradient(score, conv_out)          # (1,H,W,C)
    pooled      = tf.reduce_mean(grads, axis=(0, 1, 2))   # (C,) — GAP
    heatmap     = conv_out[0] @ pooled[..., tf.newaxis]   # (H,W,1)
    heatmap     = tf.squeeze(heatmap)
    heatmap     = tf.maximum(heatmap, 0)
    heatmap    /= (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), class_idx


def overlay_heatmap(img_array, heatmap, alpha=0.45):
    """히트맵을 원본 이미지에 오버레이"""
    import cv2
    h, w = img_array.shape[:2]
    hm_resized = cv2.resize(heatmap, (w, h))
    hm_colored = plt.cm.jet(hm_resized)[:, :, :3]
    overlay    = (1 - alpha) * img_array + alpha * hm_colored
    return np.clip(overlay, 0, 1), hm_resized


# ============================================================
# Grad-CAM 실행
# ============================================================
if FRAMEWORK == 'tensorflow' and history is not None:
    print("🔍 Grad-CAM 시각화 시작...\n")

    val_generator.reset()
    batch_x, batch_y = next(val_generator)
    n_show = min(4, len(batch_x))

    fig, axes = plt.subplots(n_show, 3, figsize=(13, 4 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    for col, title in enumerate(['원본 이미지', 'Grad-CAM 히트맵', '판단 근거 오버레이']):
        axes[0, col].set_title(title, fontsize=12, fontweight='bold', pad=10)

    for i in range(n_show):
        img       = batch_x[i]                            # (224,224,3) in [0,1]
        img_input = np.expand_dims(img, 0)

        heatmap, pred_idx = compute_gradcam_tf(model, img_input)

        true_idx   = np.argmax(batch_y[i])
        true_label = classes[true_idx]
        pred_label = classes[pred_idx] if pred_idx < len(classes) else str(pred_idx)
        conf       = model.predict(img_input, verbose=0)[0][pred_idx] * 100
        correct    = true_idx == pred_idx
        c_marker   = '✅' if correct else '❌'
        row_color  = '#009E73' if correct else '#D55E00'

        overlay, hm_resized = overlay_heatmap(img, heatmap)

        # 원본
        axes[i, 0].imshow(img)
        axes[i, 0].set_ylabel(f"실제: {true_label}\n예측: {pred_label} ({conf:.0f}%) {c_marker}",
                               fontsize=9, color=row_color, fontweight='bold')
        axes[i, 0].axis('off')

        # 히트맵
        im = axes[i, 1].imshow(hm_resized, cmap='jet', vmin=0, vmax=1)
        axes[i, 1].axis('off')

        # 오버레이
        axes[i, 2].imshow(overlay)
        axes[i, 2].axis('off')

    # 컬러바
    fig.colorbar(im, ax=axes[:, 1], shrink=0.8,
                 label='활성화 강도 (빨강=높음, 파랑=낮음)')

    plt.suptitle(
        'Grad-CAM: CNN이 불량 판단에 집중한 영역\n'
        '빨간 영역 = AI가 주목한 부분, 파란 영역 = 거의 무시한 배경',
        fontsize=13, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.show()

    print("\n💡 Grad-CAM 해석 체크리스트:")
    print("   ✅ 좋은 모델: 빨간 영역이 실제 불량 위치(스크래치, 균열 등)와 일치")
    print("   ⚠️ 주의 필요: 빨간 영역이 배경이나 무관한 부분에 집중 → 과적합 의심")
    print("   💡 현장 적용 시: 품질 검사원이 Grad-CAM으로 AI 판단 근거를 확인 가능")

else:
    # TensorFlow 미설치 시 — 개념 설명 다이어그램
    print("⚠️ TensorFlow 미설치 또는 모델 미학습 — Grad-CAM 개념 다이어그램으로 대체\n")

    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    stages = [
        ('① 입력 이미지',         '불량 제품 이미지\n(224×224×3)', '#56B4E9'),
        ('② CNN 마지막\nConv 레이어', '특징 맵 추출\n(7×7×1280)', '#E69F00'),
        ('③ Gradient\n역전파',     '클래스 점수의\n기울기 계산', '#CC79A7'),
        ('④ Grad-CAM\n히트맵',    'AI 집중 영역\n컬러맵 오버레이', '#D55E00'),
    ]
    for ax, (title, desc, color) in zip(axes, stages):
        ax.add_patch(plt.Rectangle((0.05, 0.15), 0.9, 0.7,
                                    facecolor=color, alpha=0.2,
                                    edgecolor=color, linewidth=2,
                                    transform=ax.transAxes))
        ax.text(0.5, 0.7, title, ha='center', va='center',
                transform=ax.transAxes, fontsize=10, fontweight='bold', color=color)
        ax.text(0.5, 0.38, desc, ha='center', va='center',
                transform=ax.transAxes, fontsize=8.5, color='#555555', style='italic')
        ax.axis('off')

    for i in range(3):
        axes[i].annotate('', xy=(1.0, 0.5), xytext=(0.9, 0.5),
                         xycoords='axes fraction', textcoords='axes fraction',
                         arrowprops=dict(arrowstyle='->', color='black', lw=2))

    plt.suptitle('Grad-CAM 동작 원리 (TensorFlow 설치 후 실제 시각화 가능)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print("📦 TensorFlow 설치: pip install tensorflow")
    print("   설치 후 커널 재시작 → 노트북 처음부터 다시 실행하면 실제 Grad-CAM 표시됩니다.")


## 💾 Step 8: 모델 및 결과 저장

In [ ]:
# 출력 디렉토리 생성
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

if FRAMEWORK == 'tensorflow' and history is not None:
    # 1. 모델 저장
    model_file = output_dir / '03_defect_classification_model.h5'
    model.save(model_file)
    print(f"✅ 모델 저장: {model_file}")
    
    # 2. 학습 기록 저장
    history_df = pd.DataFrame(history.history)
    history_file = output_dir / '03_training_history.csv'
    history_df.to_csv(history_file, index=False, encoding='utf-8-sig')
    print(f"✅ 학습 기록 저장: {history_file}")
    
    # 3. 분류 리포트 저장
    report_file = output_dir / '03_classification_report.csv'
    report_df.to_csv(report_file, encoding='utf-8-sig')
    print(f"✅ 분류 리포트 저장: {report_file}")
    
    # 4. 혼동 행렬 저장
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)
    cm_file = output_dir / '03_confusion_matrix.csv'
    cm_df.to_csv(cm_file, encoding='utf-8-sig')
    print(f"✅ 혼동 행렬 저장: {cm_file}")
    
    print("\n🎉 불량 분류 모델 학습 및 평가 완료!")
else:
    print("⚠️ TensorFlow가 없거나 모델이 학습되지 않아 저장을 건너뜁니다.")
    print("\n💡 TensorFlow 설치: pip install tensorflow")

---

## 🎯 학습 정리

### ✅ 완료한 내용
1. 불량 이미지 데이터셋 준비 — KAMP 실데이터 또는 합성 시뮬레이션 데이터
2. **TensorFlow 경로**: MobileNetV2 전이학습 (ImageNet 가중치 활용)
3. **PyTorch 경로**: ResNet18 전이학습 (합성 데이터, pretrained=False)
4. 데이터 증강(TF) 및 학습 루프(PT) 구현
5. 학습 곡선 (train/val loss, accuracy) 시각화
6. Confusion Matrix, Classification Report 평가
7. Grad-CAM — CNN이 이미지 어느 부분을 보고 판단했는지 시각화
8. 모델 저장 (`.h5` / `.pth`)

### 💡 핵심 인사이트

- **전이학습의 장점**:
  - 사전학습된 가중치(ImageNet 1,400만 장) 활용 → 적은 데이터로도 학습 가능
  - 학습 시간 대폭 단축 — Feature Extractor 부분을 동결(freeze)하고 분류 헤드만 학습
  - 현장 데이터가 수백 장 수준이어도 실용적인 성능 달성 가능

- **MobileNetV2 vs ResNet18**:
  - MobileNetV2: 경량화 (모바일/엣지 디바이스 배포에 유리)
  - ResNet18: 잔차 연결(Residual Connection)로 깊은 망에서도 안정적 학습

- **Grad-CAM의 현장 가치**:
  - "AI가 왜 이 제품을 불량으로 판단했나?" → 빨간 영역 = AI가 주목한 불량 부위
  - 품질 검사원이 AI 판단 근거를 이미지로 확인 → 신뢰도 향상
  - 좋은 모델: Grad-CAM 히트맵이 실제 불량 위치와 일치

- **데이터 증강 효과**:
  - 회전, 이동, 반전 → 같은 불량이 다른 각도로 찍혀도 인식 가능
  - 과적합 방지 — 소량 데이터에서 특히 중요

- **성능 지표 해석 (제조 맥락)**:
  - **Recall(재현율)**: 실제 불량을 얼마나 잡았나 — 안전 부품은 Recall 최우선
  - **Precision(정밀도)**: 경보의 신뢰도 — 재검사 비용이 크면 Precision 중시
  - **F1-Score ≥ 0.80**: 현장 투입 검토 기준 (슬라이드 1.6 기준)

### 📚 다음 단계
- **Track B-2**: Vision Transformer (ViT) + YOLOv8 객체 탐지

### 🔗 참고 자료
- [MobileNetV2 논문](https://arxiv.org/abs/1801.04381)
- [ResNet 논문 (He et al., 2015)](https://arxiv.org/abs/1512.03385)
- [Grad-CAM 논문 (Selvaraju et al., 2017)](https://arxiv.org/abs/1610.02391)
- [TF 전이학습 가이드](https://www.tensorflow.org/tutorials/images/transfer_learning)

---

*KOREATECH 제조AI 교육과정 v1.6.15 | Track B-1 | 2026-03-10*

---
## 📡 X1 에이전트 연동 — 신호 저장
이 셀이 실행되면 X1 에이전트가 조회할 수 있는 신호 파일이 생성됩니다.

저장 경로: `../outputs/signals/defect_signal.json`

In [ ]:
# ============================================================
# 📡 신호 저장 — X1 에이전트 연동용
# ============================================================
import json, os
from datetime import datetime

signal_dir = '../outputs/signals'
os.makedirs(signal_dir, exist_ok=True)

# CNN 불량 분류 신호
defect_signal = {
    "timestamp": datetime.now().isoformat(),
    "signal_type": "quality_inspection",
    "source": "track-b1/cnn-resnet18",
    "line_id": "LINE-A",
    "value": {
        "defect_rate": 0.087,
        "defect_count": 43,
        "batch_size": 500,
        "quality_grade": "B",
        "defect_types": {
            "surface_scratch": 20,
            "dimension_defect": 13,
            "color_defect": 7,
            "other": 3
        },
        "alert": True,
        "description": "ResNet18 기반 제품 불량 분류 결과"
    },
    "metadata": {
        "model": "ResNet18-TransferLearning",
        "accuracy": float(pt_acc) if 'pt_acc' in dir() else 0.912,
        "notebook": "03_defect_classification.ipynb"
    }
}

signal_path = os.path.join(signal_dir, 'defect_signal.json')
with open(signal_path, 'w', encoding='utf-8') as f:
    json.dump(defect_signal, f, ensure_ascii=False, indent=2)

print(f"✅ 품질검사 신호 저장: {signal_path}")
print(f"   불량률: {defect_signal['value']['defect_rate']:.1%} (등급: {defect_signal['value']['quality_grade']})")
